# FlashNystrom vs the sub-quadratic field: MQAR recall (LR-swept)

Apples-to-apples multi-query associative recall. Every mixer runs in ONE
environment, same backbone / seeds / data, only the attention-slot operator
changes. Backends: `sdpa`, `linear_attention`, `nystrom_reference`,
`flash_nystrom`, `flash_nystrom_tc`, `hyena`, `mamba`.

**Fairness: per-method learning-rate sweep.** Different architectures have very
different LR sensitivities, so a single fixed LR biases the comparison toward
whichever method it suits. Following the Zoology (Arora et al. 2023) convention,
we sweep the base LR per method over `LR_GRID` and report the best. The range spans 5e-4 to 1e-1: 5e-4 is the LR Hyena's paper states for its synthetics (Table A.1), and the top end extends past 3e-2 where the Nystrom variants previously peaked. A grid should bracket every method's optimum, not end on it. (Hyena's own
implicit-filter LRs, 1e-3 / 1e-5, are architectural and set internally regardless
of the base LR.)

**GPU requirement.** The `flash_nystrom` kernel and `mamba-ssm` need compute
capability >= 8.0 (A100/L4/L40S/H100). On the free **T4 (7.5)** flash_nystrom is
skipped and Mamba falls back to a ~1000x slower pure-PyTorch scan. Use an
**A100 or L4** runtime.

MQAR is bimodal, so a few seeds do not resolve the operators finely; the sweep
identifies the best LR per method and reports its seeds.

In [ ]:
import torch, subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
CC = torch.cuda.get_device_capability()
print("compute capability:", CC, "| flash_nystrom + mamba-ssm supported:", CC >= (8, 0))

In [ ]:
# Clone (with the CUTLASS submodule the kernel build needs), compile flash_nystrom
# for this arch, and install the real Mamba CUDA kernels.
import os, torch
%cd /content
!rm -rf FlashNystrom
!git clone --recursive -q https://github.com/athrva98/FlashNystrom.git
%cd /content/FlashNystrom
!pip -q install einops
CC = torch.cuda.get_device_capability()
if CC >= (8, 0):
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{CC[0]}.{CC[1]}"
    os.environ["FLASH_NYSTROM_LAX_BUILD"] = "1"  # tolerate 3rd-party header warnings
    !pip install -e . --no-build-isolation
    # Real Mamba kernels. Their setup.py imports torch, so pip build isolation
    # fails at 'getting requirements to build wheel'; --no-build-isolation uses the
    # torch already installed. causal-conv1d first (mamba-ssm depends on it).
    !pip -q install ninja packaging
    !pip install causal-conv1d --no-build-isolation
    !pip install mamba-ssm --no-build-isolation
    import flash_nystrom
    print("flash_nystrom built, version", flash_nystrom.__version__)
    from paper.mqar.baselines import _HAS_MAMBA_CUDA
    print("Mamba CUDA kernels available:", _HAS_MAMBA_CUDA)
else:
    print("sm < 8.0: skipping kernel builds; pure-PyTorch backends only (Mamba slow)")

In [ ]:
# LR sweep: each backend x each LR x seeds, run in PARALLEL on one GPU.
# An A100 hosts several independent runs at once (each peaks a few GiB); run_many
# pools them. Runs are identical to sequential -- only wall-clock improves.
import os, torch
from paper.mqar.runner import run_many

MAX_PARALLEL = 6        # concurrent runs. 80GB A100: 6-10 safe; lower if OOM /
                        # if most runs are the largest dim (they contend for SMs).
N_SEEDS = 3
LR_GRID = [5e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]
BACKENDS = ["sdpa", "linear_attention", "nystrom_reference",
            "flash_nystrom", "flash_nystrom_tc", "hyena", "mamba"]
if torch.cuda.get_device_capability() < (8, 0):
    BACKENDS = [b for b in BACKENDS if "flash_nystrom" not in b]
OUT = "runs/mqar_lrsweep"; os.makedirs(OUT, exist_ok=True)

jobs = []
for b in BACKENDS:
    for lr in LR_GRID:
        for s in range(N_SEEDS):
            out = f"{OUT}/mqar_{b}_lr{lr:g}_seed{s}.json"
            jobs.append(dict(backend=b, seed=s, dim=128, lr=lr, kappa_star=0, seq_len=256,
                             num_kv_pairs=16, num_landmarks=64, newton_iter=6,
                             batch_size=256, epochs=64, num_train=20000, num_test=2000,
                             out_json=out, log_path=out[:-5] + ".log"))
print(f"{len(jobs)} runs = {len(BACKENDS)} backends x {len(LR_GRID)} LRs x {N_SEEDS} seeds")
run_many(jobs, max_parallel=MAX_PARALLEL)

In [ ]:
import json, glob, statistics as st
from collections import defaultdict
runs = defaultdict(list)  # (backend, lr) -> [recall over seeds]
for f in glob.glob("runs/mqar_lrsweep/*.json"):
    d = json.load(open(f))
    if "lr" not in d: continue
    runs[(d["backend"], d["lr"])].append(d["best_recall"])
order = ["sdpa", "linear_attention", "nystrom_reference",
         "flash_nystrom", "flash_nystrom_tc", "hyena", "mamba"]
lrs = sorted({lr for (_, lr) in runs})

print("BEST LR per method (fair comparison: sweep LR, report best):")
print(f"  {'backend':<20} {'best_lr':>8} {'recall (mean +/- sd)':>22}  seeds")
for b in order:
    cfgs = [(lr, v) for (bb, lr), v in runs.items() if bb == b]
    if not cfgs: continue
    lr_b, v_b = max(cfgs, key=lambda x: st.mean(x[1]))
    m = st.mean(v_b); sd = st.stdev(v_b) if len(v_b) > 1 else 0.0
    seeds = ['%.1f' % x for x in sorted(v_b)]
    print(f"  {b:<20} {lr_b:>8.0e} {m:>10.2f} +/- {sd:<7.2f} {seeds}")

print("\nFull LR sweep (mean recall over seeds):")
print(f"  {'backend':<20} " + "  ".join(f"{lr:>8.0e}" for lr in lrs))
for b in order:
    row = []
    for lr in lrs:
        v = runs.get((b, lr))
        row.append(f"{st.mean(v):8.2f}" if v else f"{'--':>8}")
    print(f"  {b:<20} " + "  ".join(row))

## Hyena d>=N correctness gate (LR-swept)

Zoology shows gated convolutions like Hyena solve MQAR only once model dimension
`d >= N`. This runs Hyena at `d = 256, 512` (>= N = 256) across the LR grid (1
seed each) and reports the best recall per dimension. High recall there confirms
the vendored Hyena is faithful (it genuinely fails at d<N, not because it is
broken). Note: d=512 is slow.

In [ ]:
# Hyena d>=N gate, run in PARALLEL. (Only 12 runs, but same driver.)
import os, json, glob
from paper.mqar.runner import run_many

OUT = "runs/mqar_gate"; os.makedirs(OUT, exist_ok=True)
MAX_PARALLEL = 6
LR_GRID = [5e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1]

def heads_for(d):
    """Hold head_dim = 64 fixed (heads = d // 64). Hyena ignores the head count,
    but the attention-family kernels require head_dim in {64,128}; fixing it keeps
    the structure constant across dims and matches Zoology (num_heads=2 at d=128)."""
    return max(1, d // 64)

jobs = []
for d_model in [256, 512]:
    for lr in LR_GRID:
        out = f"{OUT}/hyena_d{d_model}_lr{lr:g}_seed0.json"
        jobs.append(dict(backend="hyena", seed=0, dim=d_model, heads=heads_for(d_model),
                         lr=lr, kappa_star=0, seq_len=256, num_kv_pairs=16, batch_size=256,
                         epochs=64, num_train=20000, num_test=2000,
                         out_json=out, log_path=out[:-5] + ".log"))
run_many(jobs, max_parallel=MAX_PARALLEL)

for d_model in [256, 512]:
    recs = [(json.load(open(f))["lr"], json.load(open(f))["best_recall"])
            for f in glob.glob(f"{OUT}/hyena_d{d_model}_lr*.json")]
    if recs:
        lr_b, r_b = max(recs, key=lambda x: x[1])
        print(f"hyena d={d_model}: best recall {r_b:.1f}% (at lr={lr_b:g})")

## DeltaNet Figure 4 reproduction (verify our operators against a published result)

Runs our Hyena and Mamba at the **exact settings of the DeltaNet paper's Figure 4**
(Yang et al. 2024, arXiv:2406.06484): sequence length 512, 64 key-value pairs,
model dimension swept over {64,128,256,512}. That figure defers to Zoology's
`iclr24_zoology_figure2` protocol, which differs from the `original_mqar` recipe
the sweep above uses:

- **uniform layers** (`--layer_layout uniform`): two layers of the mixer, no
  BaseConv (the paper: "we do not use convolutions for these experiments"). Hyena
  and Mamba carry order through their own internal conv / recurrence.
- **100k train / 3k test**, **batch 128** at seq 512, **blank non-query slots**
  (`--no-random_non_queries`), **LR = np.logspace(-4,-2,4)**, 64 epochs, 2 heads,
  early stop at 0.99. All from figure2 `configs.py`.

**Why:** it gives us a published curve to check against. Their Figure 4 (line
plot, no table): DeltaNet ~100% at every dim; Mamba ~81% at d=64 then ~100%;
Hyena ~0% at d=64/128, ~12% at d=256, ~20% at d=512. If our Hyena lands on their
Hyena curve, our Hyena is faithful (it is weak on MQAR, not broken); if our Mamba
reaches ~100% by d=128, our Mamba is validated too. Only sub-quadratic methods are
in this figure, so none get position embeddings and flash_nystrom is not included.

In [ ]:
# Faithful DeltaNet-Figure-4 sweep: Hyena + Mamba at 512/64, dims 64-512, PARALLEL.
import os, numpy as np, torch
from paper.mqar.runner import run_many

OUT = "runs/mqar_deltanet_fig4"; os.makedirs(OUT, exist_ok=True)
MAX_PARALLEL = 6        # dim-512 runs peak ~4-6 GiB; 6 is safe on 80GB.
LR_GRID  = list(np.logspace(-4, -2, 4))   # DeltaNet/figure2 grid: 1e-4 .. 1e-2
D_MODELS = [64, 128, 256, 512]
BACKENDS = ["hyena", "mamba"]
N_SEEDS  = 1

jobs = []
for b in BACKENDS:
    for d in D_MODELS:
        for lr in LR_GRID:
            for s in range(N_SEEDS):
                out = f"{OUT}/{b}_d{d}_lr{lr:.2e}_seed{s}.json"
                jobs.append(dict(backend=b, seed=s, dim=d, heads=2, lr=f"{lr:.6e}",
                                 layer_layout="uniform", random_non_queries=False,
                                 kappa_star=0, seq_len=512, num_kv_pairs=64,
                                 batch_size=128, epochs=64, early_stop_acc=0.99,
                                 num_train=100000, num_test=3000,
                                 out_json=out, log_path=out[:-5] + ".log"))
print(f"{len(jobs)} runs (Hyena+Mamba x 4 dims x {len(LR_GRID)} LRs). Faithful to "
      f"DeltaNet Fig 4 / figure2: uniform, 100k/3k, batch 128, blanks, early-stop 0.99. "
      f"Parallel x {MAX_PARALLEL}, resumable.")
run_many(jobs, max_parallel=MAX_PARALLEL)

In [ ]:
# Recall vs model dimension, best over the LR grid -- compare to DeltaNet Figure 4.
import json, glob, statistics as st
from collections import defaultdict
runs = defaultdict(list)
for f in glob.glob("runs/mqar_deltanet_fig4/*.json"):
    r = json.load(open(f)); runs[(r["backend"], r["dim"])].append(r["best_recall"])
dims = [64, 128, 256, 512]
print("Ours (best over LR), vs DeltaNet Fig 4 reference:")
print(f"  {'method':<8} " + "  ".join(f"d={d:<5}" for d in dims))
for b in ["hyena", "mamba"]:
    row = [f"{max(runs[(b,d)]):8.1f}" if runs.get((b,d)) else f"{'--':>8}" for d in dims]
    print(f"  {b:<8} " + "  ".join(row))
print("  ref hyena   ~0.0      ~0.0     ~12.0     ~20.0   (their Fig 4)")
print("  ref mamba   ~81.0    ~100.0   ~100.0    ~100.0   (their Fig 4)")

### Mamba LR in-fill (the Fig-4 grid straddles its viable band)

Mamba came back 0.0 / 0.0 / 0.0 / 99.1 across d=64..512 -- a discontinuity, not a
capacity curve. The 256/16 sweep showed why: Mamba's viable LR band is razor thin
(1e-3 -> 0.04%, **3e-3 -> 75.15%**, 1e-2 -> 0.04%, 3e-2 -> 0.05%), and the Fig-4
grid `logspace(-4,-2,4)` = 1e-4 / 4.64e-4 / 2.15e-3 / 1e-2 has **no point at 3e-3**.
So below d=512 every grid point may simply straddle the band.

This fills in {1.5e-3, 3e-3, 5e-3} at d=64/128/256 -- 9 runs. Same faithful
settings, same output dir, so the aggregation cell above picks them up and the
best-over-LR updates automatically. If recall jumps toward the reference 81-100%,
the grid was the problem; if it stays at 0.0 with 3e-3 included, the wiring in
baselines.py is the next place to look.

In [ ]:
# Mamba LR in-fill at the dims that came back dead. Same protocol as the Fig-4 cell.
import os
from paper.mqar.runner import run_many

OUT = "runs/mqar_deltanet_fig4"; os.makedirs(OUT, exist_ok=True)
MAX_PARALLEL = 6
INFILL_LRS = [1.5e-3, 3e-3, 5e-3]     # the gap between 4.64e-4 and 1e-2
D_MODELS   = [64, 128, 256]           # d=512 already works (99.1%)

jobs = []
for d in D_MODELS:
    for lr in INFILL_LRS:
        out = f"{OUT}/mamba_d{d}_lr{lr:.2e}_seed0.json"
        jobs.append(dict(backend="mamba", seed=0, dim=d, heads=2, lr=f"{lr:.6e}",
                         layer_layout="uniform", random_non_queries=False,
                         kappa_star=0, seq_len=512, num_kv_pairs=64,
                         batch_size=128, epochs=64, early_stop_acc=0.99,
                         num_train=100000, num_test=3000,
                         out_json=out, log_path=out[:-5] + ".log"))
print(f"{len(jobs)} in-fill runs: mamba x {D_MODELS} x {INFILL_LRS}")
run_many(jobs, max_parallel=MAX_PARALLEL)